In [7]:
import pandas as pd
import numpy as np
import requests
import pytz
import datetime as dt
from dotenv import load_dotenv
import os

from sklearn.linear_model import LinearRegression

import joblib

In [8]:
serial_numbers = [
    'MOD-00589',
    'MOD-00590',
    'MOD-00591',
    'MOD-00592',
    'MOD-00593',
    'MOD-00595',
]

In [9]:
# test the model on the last 1/4 of training data
def preprocess(file_path):
    sensor = pd.read_csv(file_path, parse_dates=['timestamp'])
    sensor = sensor[['timestamp_local','rh','temp','co']].dropna(subset=['co'])
    sensor.rename(columns={'timestamp_local':'time'}, inplace=True)
    
    sensor['time'] = pd.to_datetime(sensor['time'])
    est = pytz.timezone('US/Eastern')
    sensor['time'] = sensor['time'].dt.tz_convert(est)
    sensor['time'] = sensor['time'].dt.strftime('%Y-%m-%d %H:%M:%S')
    sensor['time'] = pd.to_datetime(sensor['time'])
    sensor['day'] = sensor['time'].dt.date
    sensor['dayhour'] = sensor['time'].dt.strftime('%Y-%m-%d %H')

    sensor = sensor[::-1]

    # test on first 1/4 of data
    # split_index = int(len(sensor) * 0.25)
    # sensor = sensor[:split_index]
    return sensor

In [10]:
# calculate prediction using given model
def calculate_prediction(df, model, groupBy) -> pd.DataFrame:
    if groupBy == 'dayhour':
        temp = df.groupby(groupBy).agg(
            shourlyCO=('co', lambda x: x.mean(skipna=True)),
            shourlyRH=('rh', lambda x: x.mean(skipna=True)),
            shourlyTemp=('temp', lambda x: x.mean(skipna=True))
        ).reset_index()
        
        temp['corrected_COh'] = model.predict(temp[['shourlyCO']])
        return temp[['dayhour', 'corrected_COh']]
    
    else:
        temp = df.groupby(groupBy).agg(
            sdailyCO=('co', lambda x: x.mean(skipna=True)),
            sdailyRH=('rh', lambda x: x.mean(skipna=True)),
            sdailyTemp=('temp', lambda x: x.mean(skipna=True))
        ).reset_index()
        
        temp['corrected_COd'] = model.predict(temp[['sdailyCO']])
        return temp[['day', 'corrected_COd']]

In [11]:
today = dt.datetime.today().strftime('%Y-%m-%d')
tempDay = (dt.datetime.today() - dt.timedelta(days=5)).strftime('%Y-%m-%d')

for sn in serial_numbers:
    # load model
    model_number = sn.split('-')[-1]
    hourly_model = joblib.load(f'models/hmodel_{model_number}.joblib')
    daily_model = joblib.load(f'models/dmodel_{model_number}.joblib')

    # get data
    data_fetch_url = f'https://api.quant-aq.com/device-api/v1/devices/{sn}/data-by-date/{tempDay}'
    formatted_raw_data = preprocess(f'../ShortenedData/MOD-{model_number}-RAW.csv')

    # format data

    # calculate prediction
    hourly_prediction = calculate_prediction(formatted_raw_data, hourly_model, 'dayhour')
    daily_prediction = calculate_prediction(formatted_raw_data, daily_model, 'day')

    hourly_prediction.to_csv(f'Predictions/hMOD-{model_number}-PRED.csv', index=False)
    daily_prediction.to_csv(f'Predictions/dMOD-{model_number}-PRED.csv', index=False)
    

In [12]:
hourly_model = joblib.load(f'models/hmodel_00589.joblib')
daily_model = joblib.load(f'models/dmodel_00589.joblib')

# get data
formatted_raw_data = preprocess(f'../presentationData/current-MOD-00589-RAW.csv')

# calculate prediction
hourly_prediction = calculate_prediction(formatted_raw_data, hourly_model, 'dayhour')
daily_prediction = calculate_prediction(formatted_raw_data, daily_model, 'day')
hourly_prediction.to_csv(f'../presentationData/hMOD-00589-PRED.csv', index=False)
daily_prediction.to_csv(f'../presentationData/dMOD-00589-PRED.csv', index=False)